# NER on Titles — Headline Entity Extraction

Variant of the corpus-wide NER pipeline that runs the Flair `flair/ner-german-large` model on **article titles only** instead of full bodies. Used as a robustness check on the entity frequencies feeding into the mainstream-media filter list — titles are short, high-signal, and not affected by article-length differences across outlets.

In [ ]:
import json

import pandas as pd
from IPython.display import display

df = pd.read_csv("df_combined.csv")

print(f"Shape: {df.shape}")
display(df.head(3))


## Step 1: Prepare the text

Clean `Title` and `Text`, keep a stable `row_id`, and combine title plus article body into one `document_text` field.

In [2]:
import hashlib
import html
import re
from collections import defaultdict

# --------------------------------------------------
# 1. CLEAN THE TEXT BEFORE NER
# --------------------------------------------------

SOURCE_BOILERPLATE = {
    "Antispiegel": [
        r"Ende der Übersetzung\.?$",
        r"Man kann .*?(?:auf X|auf Telegram).*? folgen\.?$",
    ],
    "Compact": [
        r"Hier bestellen!?\.?$",
        r"Jetzt bestellen!?\.?$",
        r"Hier mehr erfahren\.?$",
        r"zum einmaligen Angebotspreis .*?$",
    ],
    "Deutschlandkurier": [
        r"Bitte beachten Sie, dass dabei Daten an Drittanbieter weitergegeben werden\..*$",
    ],
    "Nius": [
        r"Hier geht['’]s direkt zur Aufzeichnung.*$",
        r"Ab sofort auch in .*?$",
    ],
    "RT_de": [
        r"Ende der Übersetzung\.?$",
        r"Man kann .*?(?:auf X|auf Telegram).*? folgen\.?$",
    ],
    "Tagesschau": [
        r"Über dieses Thema berichtete .*?(?:Uhr\.?)?$",
        r"\|\s*\d{2}\.\d{2}\.\d{4}\s*\|\s*\d{2}:\d{2}\s*Uhr$",
    ],
    "Tichys_Einblick": [
        r"^Ich bin damit einverstanden, dass mir von (?:Youtube|Twitter) .*? werden\.\s*",
        r"Mit Ihrem Einkauf im TE-Shop .*?$",
        r"unterstützen Sie den unabhängigen Journalismus von Tichys Einblick!.*$",
    ],
}

def normalize_unicode(text: str) -> str:
    text = html.unescape(str(text))
    replacements = {
        "\u00a0": " ",
        "\u200b": " ",
        "’": "'",
        "‘": "'",
        "´": "'",
        "`": "'",
        "“": '"',
        "”": '"',
        "–": "-",
        "—": "-",
        "…": "...",
        "￼": " ",
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    return text

def strip_source_boilerplate(text: str, source: str) -> str:
    for pattern in SOURCE_BOILERPLATE.get(source, []):
        text = re.sub(pattern, " ", text, flags=re.IGNORECASE)
    return text

def clean_article_text(text: str, source: str) -> str:
    text = normalize_unicode(text)
    text = re.sub(r"^#+\s*", "", text)
    text = strip_source_boilerplate(text, source)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"\b\S+@\S+\b", " ", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

work = df.copy()

work["Title_clean"] = [
    clean_article_text(title, source)
    for title, source in zip(work["Title"].fillna(""), work["source"].fillna(""))
]

work["Text_clean"] = [
    clean_article_text(text, source)
    for text, source in zip(work["Text"].fillna(""), work["source"].fillna(""))
]

# For the full-text run, use the body text as the main NER input.
# If you want title context too, prepend the title WITHOUT "Title:" / "Text:" labels.
work = work[work["Title_clean"].str.len() > 0].copy()
work["document_text"] = work["Title_clean"]
# optional:
# work["document_text"] = (work["Title_clean"] + "\n\n" + work["Text_clean"]).str.strip()

# Deduplicate only for inference speed, not for the final analysis.
work["doc_hash"] = work["document_text"].map(
    lambda x: hashlib.md5(x.encode("utf-8")).hexdigest()
)
run_df = work.drop_duplicates("doc_hash").copy()

print(f"Rows kept for analysis: {len(work):,}")
print(f"Unique texts sent to NER: {len(run_df):,}")

Rows kept for analysis: 20,440
Unique texts sent to NER: 20,426


## Step 2: Load the Flair NER model

If Flair is not installed yet, run the install command once, restart the kernel, and then run the cell below.

```python
%pip install flair
```

In [3]:
!pip install flair

In [4]:
from flair.models import SequenceTagger
from flair.splitter import SegtokSentenceSplitter

MODEL_NAME = "flair/ner-german-large"
tagger = SequenceTagger.load(MODEL_NAME)
splitter = SegtokSentenceSplitter()

print(f"Loaded model: {MODEL_NAME}")

2026-03-24 08:51:55,321 SequenceTagger predicts: Dictionary with 20 tags: <unk>, O, B-PER, E-PER, S-LOC, B-MISC, I-MISC, E-MISC, S-PER, B-ORG, E-ORG, S-ORG, I-ORG, B-LOC, E-LOC, S-MISC, I-PER, I-LOC, <START>, <STOP>
Loaded model: flair/ner-german-large


## Step 3: Run NER on the corpus

Each article is split into sentences first. Then the model predicts NER tags for those sentences. The extracted entity spans are stored back on article level.

In [5]:
import os
import torch
import flair

# Allow fallback to CPU for ops not implemented on MPS
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

if torch.backends.mps.is_available():
    flair.device = torch.device("mps")
    device_name = "MPS"
elif torch.cuda.is_available():
    flair.device = torch.device("cuda")
    device_name = "CUDA"
else:
    flair.device = torch.device("cpu")
    device_name = "CPU"

# Reload model so it is placed on the selected device
tagger = SequenceTagger.load(MODEL_NAME)
tagger.to(flair.device)

print(f"Using device: {device_name} ({flair.device})")
print(f"Model loaded: {MODEL_NAME}")

2026-03-24 08:52:11,137 SequenceTagger predicts: Dictionary with 20 tags: <unk>, O, B-PER, E-PER, S-LOC, B-MISC, I-MISC, E-MISC, S-PER, B-ORG, E-ORG, S-ORG, I-ORG, B-LOC, E-LOC, S-MISC, I-PER, I-LOC, <START>, <STOP>
Using device: MPS (mps)
Model loaded: flair/ner-german-large


In [6]:
KEEP_LABELS = {"PER", "ORG", "LOC"}
MIN_SCORE = 0.85

def extract_entities(text):
    if not isinstance(text, str) or not text.strip():
        return []

    sentences = splitter.split(text)
    if not sentences:
        return []

    tagger.predict(sentences, mini_batch_size=1)

    entities = []
    for sentence in sentences:
        for span in sentence.get_spans("ner"):
            label = span.tag
            score = float(span.score)

            if label not in KEEP_LABELS:
                continue
            if score < MIN_SCORE:
                continue

            entities.append(
                {
                    "entity_raw": span.text,
                    "entity": span.text,
                    "label": label,
                    "score": score,
                    # keep start/end only if you really need them later
                    # "start": sentence.start_position + span.start_position,
                    # "end": sentence.start_position + span.end_position,
                }
            )

    return entities


In [7]:
# NER on texts

all_entities = []
for idx, text in enumerate(run_df["document_text"], start=1):
    all_entities.append(extract_entities(text))
    if idx % 100 == 0 or idx == len(run_df):
        print(f"Processed {idx}/{len(run_df)} unique texts")

run_df["entities"] = all_entities

work = work.drop(columns=["entities"], errors="ignore").merge(
    run_df[["doc_hash", "entities"]],
    on="doc_hash",
    how="left",
)

work["entity_count"] = work["entities"].str.len()


Processed 100/20426 unique texts
Processed 200/20426 unique texts
Processed 300/20426 unique texts
Processed 400/20426 unique texts
Processed 500/20426 unique texts
Processed 600/20426 unique texts
Processed 700/20426 unique texts
Processed 800/20426 unique texts
Processed 900/20426 unique texts
Processed 1000/20426 unique texts
Processed 1100/20426 unique texts
Processed 1200/20426 unique texts
Processed 1300/20426 unique texts
Processed 1400/20426 unique texts
Processed 1500/20426 unique texts
Processed 1600/20426 unique texts
Processed 1700/20426 unique texts
Processed 1800/20426 unique texts
Processed 1900/20426 unique texts
Processed 2000/20426 unique texts
Processed 2100/20426 unique texts
Processed 2200/20426 unique texts
Processed 2300/20426 unique texts
Processed 2400/20426 unique texts
Processed 2500/20426 unique texts
Processed 2600/20426 unique texts
Processed 2700/20426 unique texts
Processed 2800/20426 unique texts
Processed 2900/20426 unique texts
Processed 3000/20426 un

In [8]:
#Normalize Entity String list after NER


SURFACE_ALIAS_MAP = {
    # spelling / orthography variants
    "selenskij": "Selenskyj",
    "selenskijs": "Selenskyj",
    "selensky": "Selenskyj",
    "selenskys": "Selenskyj",
    "selenskyjs": "Selenskyj",
    "selenski": "Selenskyj",
    "weissrussland": "Weißrussland",
    "weissrusslands": "Weißrussland",
    "von der leyen": "Von der Leyen",
    "ursula von der leyen": "Ursula von der Leyen",
    # German case-inflected multiword entities
    "europäischen union": "Europäische Union",
    "nahen osten": "Naher Osten",
    "nahen ostens": "Naher Osten",
    "weißen haus": "Weißes Haus",
}

# Use this only for HIGH-PRECISION people that are unambiguous in your corpus.
PERSON_CANONICAL_MAP = {
    "trump": "Donald Trump",
    "donald trump": "Donald Trump",
    "merz": "Friedrich Merz",
    "friedrich merz": "Friedrich Merz",
    "habeck": "Robert Habeck",
    "robert habeck": "Robert Habeck",
    "putin": "Wladimir Putin",
    "wladimir putin": "Wladimir Putin",
    "musk": "Elon Musk",
    "elon musk": "Elon Musk",
    "selenskyj": "Wolodymyr Selenskyj",
    "wolodymyr selenskyj": "Wolodymyr Selenskyj",
}

BLOCKLIST = {"i", "j", "l", "x"}
NO_STRIP_FINAL_S = {"Hamas", "Reuters", "Focus", "Taurus"}

def normalize_entity_surface(text: str) -> str:
    text = normalize_unicode(text)
    text = re.sub(r"\s+", " ", text).strip()
    text = text.strip(" \"'.,;:!?()[]{}")
    return text

def strip_german_genitive(entity: str, label: str) -> str:
    # remove trailing apostrophe forms: Merz' -> Merz
    entity = re.sub(r"[']+$", "", entity).strip()

    # remove trailing genitive s only for PER / LOC
    # this catches Trump/Trumps, Habeck/Habecks, Israel/Israels, Russland/Russlands
    if (
        label in {"PER", "LOC"}
        and len(entity) >= 5
        and entity.endswith("s")
        and not entity.isupper()
        and entity not in NO_STRIP_FINAL_S
    ):
        entity = entity[:-1]

    return entity

def normalize_entity(entity: str, label: str = "") -> str | None:
    entity = normalize_entity_surface(entity)

    if not entity or len(entity) < 2:
        return None
    if entity.casefold() in BLOCKLIST:
        return None

    entity = strip_german_genitive(entity, label)

    surface_match = SURFACE_ALIAS_MAP.get(entity.casefold())
    if surface_match:
        entity = surface_match

    if label == "PER":
        person_match = PERSON_CANONICAL_MAP.get(entity.casefold())
        if person_match:
            entity = person_match

    entity = strip_german_genitive(entity, label)

    if not entity or len(entity) < 2:
        return None
    if entity.casefold() in BLOCKLIST:
        return None

    return entity

def expand_person_surnames_within_doc(entity_list: list[dict]) -> list[dict]:
    # If a document contains both "Donald Trump" and "Trump",
    # collapse the short form to the full form within that document only.
    surname_to_full = defaultdict(set)

    for e in entity_list:
        if e.get("label") != "PER":
            continue
        parts = e.get("entity", "").split()
        if len(parts) >= 2:
            surname_to_full[parts[-1].casefold()].add(e["entity"])

    expanded = []
    for e in entity_list:
        item = dict(e)
        if item.get("label") == "PER":
            candidates = surname_to_full.get(item["entity"].casefold(), set())
            if len(candidates) == 1:
                item["entity"] = next(iter(candidates))
        expanded.append(item)

    return expanded

def normalize_entity_list(entity_list: list[dict]) -> list[dict]:
    cleaned = []

    for e in entity_list or []:
        canonical = normalize_entity(e.get("entity", ""), e.get("label", ""))
        if canonical is None:
            continue

        item = dict(e)
        item["entity"] = canonical
        cleaned.append(item)

    cleaned = expand_person_surnames_within_doc(cleaned)
    return cleaned

work["entities"] = work["entities"].apply(normalize_entity_list)
work["entity_count"] = work["entities"].str.len()

print("Normalization complete.")


Normalization complete.


In [9]:
#Summary Table

mentions = (
    work[["row_id", "source", "entities"]]
    .explode("entities")
    .dropna(subset=["entities"])
    .reset_index(drop=True)
)

entity_parts = mentions["entities"].apply(pd.Series)
mentions = pd.concat([mentions.drop(columns="entities"), entity_parts], axis=1)

mentions["entity"] = mentions["entity"].fillna("").astype(str).str.strip()
mentions = mentions[mentions["entity"] != ""].copy()

entity_summary = (
    mentions.groupby(["entity", "label"])
    .agg(
        mentions=("entity", "size"),
        documents=("row_id", "nunique"),
        sources=("source", "nunique"),
        mean_score=("score", "mean"),
    )
    .sort_values(["documents", "mentions"], ascending=False)
    .reset_index()
)

display(entity_summary.head(50))


,entity,label,mentions,documents,sources,mean_score
0,Donald Trump,PER,1138,1136,7,0.999936
1,Deutschland,LOC,798,794,7,0.999996
2,Russland,LOC,764,761,7,0.999986
3,Friedrich Merz,PER,697,695,7,0.999933
4,AfD,ORG,613,602,7,0.999631
5,USA,LOC,595,595,7,0.999446
6,Ukraine,LOC,523,523,7,0.999996
7,EU,ORG,359,359,7,0.994774
8,Wladimir Putin,PER,351,351,7,0.999976
9,Europa,LOC,345,342,7,0.999553


#### Save Results

In [10]:
all_entities = (
    mentions.groupby("entity")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

all_entities.to_csv("all_entities_counts.csv", index=False, encoding="utf-8")
print("Saved to all_entities_counts.csv")


Saved to all_entities_counts.csv
